# MedPulse India: XGBoost Surgeon Risk Classifier Training
This notebook loads the real-world dataset from `medpulse.db`, pre-processes the dynamic and static features, shifts the target class to predict 3 hours ahead, and trains an XGBoost surge classifier.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sqlalchemy import create_engine
import xgboost as xgb

sys.path.append(os.path.abspath('.'))
from app.core.database import DB_PATH
from app.ml.train_model import FEATURE_COLUMNS, MODEL_PATH, FEATURES_PATH

print("DB Path:", DB_PATH)
print("Model Path:", MODEL_PATH)

## Step 1: Load and Process Data from SQLite Warehouse
We join regions, weather logs, and hospital logs. We compute the bed occupancy stress ratio 3 hours in the future to define the target risk classes (Stable, Elevated, High, Critical).

In [ ]:
engine = create_engine(f"sqlite:///{DB_PATH}")
regions_df = pd.read_sql("SELECT * FROM regions", engine)
weather_df = pd.read_sql("SELECT * FROM weather_aqi_logs", engine)
hospital_df = pd.read_sql("SELECT * FROM hospital_logs", engine)

logs_df = pd.merge(weather_df, hospital_df, on=["region_id", "timestamp"]).sort_values(by=["region_id", "timestamp"])
full_df = pd.merge(logs_df, regions_df, left_on="region_id", right_on="id")

# Lagged features
full_df["lagged_admissions"] = full_df["admitted_count"]
full_df["lagged_occupancy"] = full_df["beds_occupied"]
full_df["lagged_icu_load"] = full_df["icu_load"]

# Target calculation
full_df["total_beds_capacity"] = (full_df["baseline_beds"] * (full_df["population"] / 1000)).astype(int).replace(0, 1)
full_df["occupancy_ratio"] = full_df["active_patients"] / full_df["total_beds_capacity"]

def categorize_risk(ratio):
    if ratio <= 0.60: return 0
    elif ratio <= 0.80: return 1
    elif ratio <= 0.95: return 2
    else: return 3

full_df["risk_category"] = full_df["occupancy_ratio"].apply(categorize_risk)
full_df["target_risk"] = full_df.groupby("region_id")["risk_category"].shift(-1)
full_df = full_df.dropna(subset=["target_risk"])
full_df["target_risk"] = full_df["target_risk"].astype(int)

print(f"Dataset processed. Rows: {len(full_df)}")

## Step 2: Split and Train XGBoost surge classifier
We train an XGBoost multi-classifier using balanced class sample weights to correct any imbalance.

In [ ]:
X = full_df[FEATURE_COLUMNS]
y = full_df["target_risk"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

from sklearn.utils.class_weight import compute_sample_weight
sample_weight = compute_sample_weight(class_weight='balanced', y=y_train)

print("Training XGBoost Classifier...")
model = xgb.XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42, eval_metric="mlogloss")
model.fit(X_train, y_train, sample_weight=sample_weight)

y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"\nAccuracy: {acc:.4f}")
print(classification_report(y_test, y_pred, target_names=["Stable", "Elevated", "High", "Critical"]))

## Step 3: Save Model Assets
Saves the trained model and feature list back to their respective joblib paths.

In [ ]:
os.makedirs(os.path.dirname(MODEL_PATH), exist_ok=True)
joblib.dump(model, MODEL_PATH)
joblib.dump(FEATURE_COLUMNS, FEATURES_PATH)
print("Model assets saved successfully!")